# Train on a consent-filtered split

Run this notebook in a kernel with `comot[huggingface]` installed. The three fictional rows have a numeric feature and a regression target. Everything runs locally.

In [ ]:
from comot import Comot, ConsentRefused
from comot.huggingface import filter_allowed, register_split
from datasets import Dataset

ledger = Comot()
ids, codes = [], []
for i, training in enumerate([True, False, True]):
    participant, code = ledger.create_participant(actor="collector")
    ledger.record_consent(participant, "demo", {"record": True, "research": True, "model_training": training},
                          script_version="demo", language="pcm", capture_method="spoken_yes_no",
                          teach_back_score="5/5", device_id="demo", actor="collector")
    source = ledger.register_recording(participant, "demo", content=f"fictional-{i}".encode(), actor="collector")
    ids.append(source.artifact_id)
    codes.append(code)
rows = Dataset.from_dict({"artifact_id": ids, "x": [1, 2, 3], "y": [2, 99, 6]})
research = register_split(ledger, rows, "research", actor="pipeline")

## Select training rows and fit a model

Register the selected rows as a separate dataset version. Check its current consent immediately before fitting a least-squares line through the origin.

In [ ]:
train = filter_allowed(ledger, research.dataset, "model_training", actor="pipeline")
ledger.check_training_run([train.artifact.artifact_id], actor="pipeline")
x, y = train.dataset["x"], train.dataset["y"]
slope = sum(a*b for a, b in zip(x, y)) / sum(a*a for a in x)
model = ledger.register("model_version", [train.artifact.artifact_id], "model_training", actor="pipeline")
assert len(x) == 2 and slope == 2.0
print(f"Fitted y = {slope}x on {len(x)} rows")

## Withdraw one contributor

The prior dataset and model retain their lineage. Trying to register a fresh copy of the withdrawn row reports the reasons for refusal.

In [ ]:
ledger.withdraw("demo", code=codes[0], channel="participant_code", actor="collector")
assert not ledger.check_use(model.artifact_id, "model_training").allowed
try:
    register_split(ledger, rows.select([0]), "model_training", actor="pipeline")
except ConsentRefused as exc:
    print(exc.blockers)
    assert any(key == ids[0] for key, reason in exc.blockers)
else:
    raise AssertionError("the withdrawn row should have been refused")